# Cleaned analysis notebook

This copy preserves the original analysis/calculation cells. Personal machine paths have been replaced with project-relative paths.
Place input workbooks and data in `data/`; generated files are written to `outputs/`. Create these folders before running.
Saved cell outputs were cleared so results are regenerated when the notebook is run.


# Individual structure/level results across all 210 cases HD95 and ASD

In [ ]:
!pip install medpy

In [ ]:
# ============================================================
# SPIDER nnU-Net
# HD95 + ASD analysis
#
# Outputs:
# 1. Overall vertebrae/IVD HD95 + ASD
# 2. Level-wise HD95 + ASD
# 3. Mean ± SD
# 4. Median
# 5. Q1 / Q3
# 6. P90
# 7. N
# 8. Five-fold summaries
# 9. Supplementary boxplots
# ============================================================


In [ ]:
import os
import re
import glob
import numpy as np
import pandas as pd
import SimpleITK as sitk
from medpy.metric import binary
import matplotlib.pyplot as plt

# ============================================================
# FILE / DIRECTORY PATHS
# ============================================================

FILE_PATH = "data/Original_dice_all_fold.xlsx"

GT_DIR = "data/masks"

PRED_DIR = "data/predictions"

OUTPUT_DIR = "outputs"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("GT directory:", GT_DIR)
print("Prediction directory:", PRED_DIR)
print("Output directory:", OUTPUT_DIR)

Build the GT and prediction indexes

In [ ]:
# ============================================================
# CREATE GT AND PREDICTION FILE INDEX
# ============================================================

gt_files = glob.glob(os.path.join(GT_DIR, "*_t2.mha"))
pred_files = glob.glob(os.path.join(PRED_DIR, "*.nii"))

print("GT files found:", len(gt_files))
print("Prediction files found:", len(pred_files))

In [ ]:
def extract_case_id(filepath):
    filename = os.path.basename(filepath)

    match = re.search(r'(\d+)', filename)

    if match:
        return int(match.group(1))

    return None

In [ ]:
gt_index = {}

for f in gt_files:
    case_id = extract_case_id(f)
    if case_id is not None:
        gt_index[case_id] = f


pred_index = {}

for f in pred_files:
    case_id = extract_case_id(f)
    if case_id is not None:
        pred_index[case_id] = f


print("GT indexed cases:", len(gt_index))
print("Prediction indexed cases:", len(pred_index))

common_cases = sorted(
    set(gt_index.keys()) & set(pred_index.keys())
)

print("Common cases:", len(common_cases))

print("First 10 case IDs:", common_cases[:10])

3. Check the actual labels

In [ ]:
# ============================================================
# CHECK ACTUAL LABELS
# ============================================================

test_case = common_cases[0]

gt_img = sitk.ReadImage(gt_index[test_case])
pred_img = sitk.ReadImage(pred_index[test_case])

gt_test = sitk.GetArrayFromImage(gt_img)
pred_test = sitk.GetArrayFromImage(pred_img)

print("Test case:", test_case)

print("\nGT unique labels:")
print(np.unique(gt_test))

print("\nPrediction unique labels:")
print(np.unique(pred_test))

print("\nGT spacing:")
print(gt_img.GetSpacing())

print("\nPrediction spacing:")
print(pred_img.GetSpacing())

4. Label mapping

In [ ]:
# ============================================================
# ANATOMICAL LABEL MAPPING
# ============================================================

VERTEBRA_MAP = {
    "L1": 5,
    "L2": 4,
    "L3": 3,
    "L4": 2,
    "L5": 1
}

GT_IVD_MAP = {
    "T12/L1": 206,
    "L1/L2": 205,
    "L2/L3": 204,
    "L3/L4": 203,
    "L4/L5": 202,
    "L5/S1": 201
}

PRED_IVD_MAP = {
    "T12/L1": 11,
    "L1/L2": 10,
    "L2/L3": 9,
    "L3/L4": 8,
    "L4/L5": 7,
    "L5/S1": 6
}

5. Metric function

In [ ]:
# ============================================================
# HD95 + ASD METRIC FUNCTION
# ============================================================

def calculate_distance_metrics(gt_mask, pred_mask, spacing):

    gt_mask = np.asarray(gt_mask).astype(bool)
    pred_mask = np.asarray(pred_mask).astype(bool)

    if gt_mask.any() and pred_mask.any():

        dice = binary.dc(pred_mask, gt_mask)

        hd95 = binary.hd95(
            pred_mask,
            gt_mask,
            voxelspacing=spacing,
            connectivity=1
        )

        assd = binary.assd(
            pred_mask,
            gt_mask,
            voxelspacing=spacing,
            connectivity=1
        )

        return dice, hd95, assd

    return np.nan, np.nan, np.nan

6. Calculate the level-wise results

In [ ]:
# ============================================================
# FASTER LEVEL-WISE HD95 + ASD
# ============================================================

level_results = []

total_cases = len(common_cases)

for case_idx, case_id in enumerate(common_cases, start=1):

    print(f"Processing case {case_idx}/{total_cases}: {case_id}")

    # --------------------------------------------------------
    # READ IMAGES ONCE
    # --------------------------------------------------------

    gt_img = sitk.ReadImage(gt_index[case_id])
    pred_img = sitk.ReadImage(pred_index[case_id])

    gt = sitk.GetArrayFromImage(gt_img)
    pred = sitk.GetArrayFromImage(pred_img)

    # --------------------------------------------------------
    # CHECK SHAPE
    # --------------------------------------------------------

    if gt.shape != pred.shape:
        print(
            f"  WARNING: shape mismatch "
            f"{gt.shape} vs {pred.shape}"
        )
        continue

    # NumPy array order = (z, y, x)
    spacing = tuple(reversed(gt_img.GetSpacing()))

    # --------------------------------------------------------
    # VERTEBRAE
    # --------------------------------------------------------

    for level, label in VERTEBRA_MAP.items():

        gt_mask = (gt == label)
        pred_mask = (pred == label)

        dice, hd95, assd = calculate_distance_metrics(
            gt_mask,
            pred_mask,
            spacing
        )

        level_results.append({
            "case_id": case_id,
            "structure_type": "Vertebra",
            "level": level,
            "Dice": dice,
            "HD95_mm": hd95,
            "ASSD_mm": assd
        })

    # --------------------------------------------------------
    # IVDS
    # --------------------------------------------------------

    for level in GT_IVD_MAP:

        gt_mask = (gt == GT_IVD_MAP[level])
        pred_mask = (pred == PRED_IVD_MAP[level])

        dice, hd95, assd = calculate_distance_metrics(
            gt_mask,
            pred_mask,
            spacing
        )

        level_results.append({
            "case_id": case_id,
            "structure_type": "IVD",
            "level": level,
            "Dice": dice,
            "HD95_mm": hd95,
            "ASSD_mm": assd
        })

    # --------------------------------------------------------
    # SAVE EVERY 10 CASES
    # --------------------------------------------------------

    if case_idx % 10 == 0:

        temp_df = pd.DataFrame(level_results)

        temp_file = os.path.join(
            OUTPUT_DIR,
            "levelwise_HD95_ASSD_progress.csv"
        )

        temp_df.to_csv(temp_file, index=False)

        print(
            f"  Saved progress: {len(temp_df)} rows"
        )


# ============================================================
# FINAL DATAFRAME
# ============================================================

level_df = pd.DataFrame(level_results)

print("\n========================================")
print("COMPLETED")
print("========================================")
print("Total rows:", len(level_df))

display(level_df.head())

7. Calculate Median/ IQR/ P90

In [ ]:
# ============================================================
# MEDIAN / IQR / P90
# ============================================================

def summarize_metric(series):

    series = series.dropna()

    return pd.Series({
        "N": len(series),
        "Mean": series.mean(),
        "SD": series.std(),
        "Median": series.median(),
        "Q1": series.quantile(0.25),
        "Q3": series.quantile(0.75),
        "P90": series.quantile(0.90)
    })


hd95_summary = (
    level_df
    .groupby(["structure_type", "level"])["HD95_mm"]
    .apply(summarize_metric)
    .unstack()
    .reset_index()
)

asd_summary = (
    level_df
    .groupby(["structure_type", "level"])["ASSD_mm"]
    .apply(summarize_metric)
    .unstack()
    .reset_index()
)

print("HD95:")
display(hd95_summary)

print("ASSD:")
display(assd_summary)

7. Save the results

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

level_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "HD95_ASD_all_level_observations.csv"
    ),
    index=False
)

hd95_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "HD95_levelwise_summary.csv"
    ),
    index=False
)

asd_summary.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "ASD_levelwise_summary.csv"
    ),
    index=False
)

print("Files saved to:")
print(OUTPUT_DIR)

Calculate Overall and level-wise HD95/ASD distributions

In [ ]:
# Keep only observations with valid distance metrics
valid_distance_df = df.dropna(subset=["HD95_mm", "ASD_mm"]).copy()

print("Total valid distance observations:", len(valid_distance_df))
print("Total excluded observations:", len(df) - len(valid_distance_df))


# ---------------------------------------------------------
# 3A. Overall distribution by structure type
# ---------------------------------------------------------

overall_distribution = (
    valid_distance_df
    .groupby("structure_type")
    .agg(
        N=("HD95_mm", "count"),

        HD95_mean=("HD95_mm", "mean"),
        HD95_SD=("HD95_mm", "std"),
        HD95_median=("HD95_mm", "median"),
        HD95_Q1=("HD95_mm", lambda x: x.quantile(0.25)),
        HD95_Q3=("HD95_mm", lambda x: x.quantile(0.75)),
        HD95_P90=("HD95_mm", lambda x: x.quantile(0.90)),

        ASD_mean=("ASD_mm", "mean"),
        ASD_SD=("ASD_mm", "std"),
        ASD_median=("ASD_mm", "median"),
        ASD_Q1=("ASD_mm", lambda x: x.quantile(0.25)),
        ASD_Q3=("ASD_mm", lambda x: x.quantile(0.75)),
        ASD_P90=("ASD_mm", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

overall_distribution["HD95_IQR"] = (
    overall_distribution["HD95_Q3"]
    - overall_distribution["HD95_Q1"]
)

overall_distribution["ASD_IQR"] = (
    overall_distribution["ASD_Q3"]
    - overall_distribution["ASD_Q1"]
)

print("\nOVERALL DISTRIBUTION")
display(overall_distribution)


Level-wise distribution

In [ ]:
level_distribution = (
    valid_distance_df
    .groupby(["structure_type", "level"])
    .agg(
        N=("HD95_mm", "count"),

        HD95_mean=("HD95_mm", "mean"),
        HD95_SD=("HD95_mm", "std"),
        HD95_median=("HD95_mm", "median"),
        HD95_Q1=("HD95_mm", lambda x: x.quantile(0.25)),
        HD95_Q3=("HD95_mm", lambda x: x.quantile(0.75)),
        HD95_P90=("HD95_mm", lambda x: x.quantile(0.90)),

        ASD_mean=("ASD_mm", "mean"),
        ASD_SD=("ASD_mm", "std"),
        ASD_median=("ASD_mm", "median"),
        ASD_Q1=("ASD_mm", lambda x: x.quantile(0.25)),
        ASD_Q3=("ASD_mm", lambda x: x.quantile(0.75)),
        ASD_P90=("ASD_mm", lambda x: x.quantile(0.90)),
    )
    .reset_index()
)

level_distribution["HD95_IQR"] = (
    level_distribution["HD95_Q3"]
    - level_distribution["HD95_Q1"]
)

level_distribution["ASD_IQR"] = (
    level_distribution["ASD_Q3"]
    - level_distribution["ASD_Q1"]
)

print("\nLEVEL-WISE DISTRIBUTION")
display(level_distribution)

11. HD95 distribution

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------

file_path = "outputs/levelwise_HD95_ASD_progress.csv"

df = pd.read_csv(file_path)

print("Rows:", len(df))
print(df.columns.tolist())

# ------------------------------------------------------------
# 2. Keep valid HD95 observations
# ------------------------------------------------------------

plot_df = df.dropna(subset=["HD95_mm"]).copy()

# Standardize structure names if necessary
plot_df["structure_type"] = plot_df["structure_type"].replace({
    "Vertebra": "Vertebra",
    "Vertebrae": "Vertebra",
    "IVD": "IVD"
})

# ------------------------------------------------------------
# 3. Define anatomical order
# ------------------------------------------------------------

level_order = [
    "L1", "L2", "L3", "L4", "L5",
    "T12/L1", "L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"
]

# Keep only levels actually present
level_order = [x for x in level_order if x in plot_df["level"].unique()]

# ------------------------------------------------------------
# 4. Plot
# ------------------------------------------------------------

plt.figure(figsize=(13, 7))

ax = sns.boxplot(
    data=plot_df,
    x="level",
    y="HD95_mm",
    order=level_order,
    hue="structure_type",
    dodge=False,
    showfliers=True,
    linewidth=1.2
)

plt.xlabel("Anatomical level", fontsize=13)
plt.ylabel("HD95 (mm)", fontsize=13)
plt.title("HD95 Distribution by Anatomical Level", fontsize=15)

plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()

# Save high-resolution figure
plt.savefig(
    "outputs/HD95_distribution_by_level.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# HD95 distribution: Vertebrae vs IVDs
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = df.dropna(subset=["HD95_mm"]).copy()

# ------------------------------------------------------------
# Separate structure types
# ------------------------------------------------------------

vertebra_levels = ["L1", "L2", "L3", "L4", "L5"]

ivd_levels = [
    "T12/L1",
    "L1/L2",
    "L2/L3",
    "L3/L4",
    "L4/L5",
    "L5/S1"
]

vertebra_df = plot_df[
    plot_df["structure_type"].isin(["Vertebra", "Vertebrae"]) &
    plot_df["level"].isin(vertebra_levels)
]

ivd_df = plot_df[
    (plot_df["structure_type"] == "IVD") &
    plot_df["level"].isin(ivd_levels)
]

# ------------------------------------------------------------
# Create figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

# -------------------------
# Vertebrae
# -------------------------

sns.boxplot(
    data=vertebra_df,
    x="level",
    y="HD95_mm",
    order=vertebra_levels,
    ax=axes[0],
    showfliers=True,
    linewidth=1.2
)

axes[0].set_title("Vertebrae", fontsize=14)
axes[0].set_xlabel("Vertebral level", fontsize=12)
axes[0].set_ylabel("HD95 (mm)", fontsize=12)
axes[0].grid(axis="y", linestyle="--", alpha=0.3)

# -------------------------
# IVDs
# -------------------------

sns.boxplot(
    data=ivd_df,
    x="level",
    y="HD95_mm",
    order=ivd_levels,
    ax=axes[1],
    showfliers=True,
    linewidth=1.2
)

axes[1].set_title("Intervertebral discs", fontsize=14)
axes[1].set_xlabel("Disc level", fontsize=12)
axes[1].set_ylabel("HD95 (mm)", fontsize=12)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

plt.suptitle(
    "Distribution of HD95 Across Anatomical Levels",
    fontsize=16,
    y=1.02
)

plt.tight_layout()

plt.savefig(
    "outputs/HD95_distribution_by_level_separate.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

13. ASD distribution

In [ ]:
# ============================================================
# ASD distribution: Vertebrae vs IVDs
# ============================================================

fig, axes = plt.subplots(
    1, 2,
    figsize=(14, 6)
)

# -------------------------
# Vertebrae
# -------------------------

sns.boxplot(
    data=vertebra_df,
    x="level",
    y="ASD_mm",
    order=vertebra_levels,
    ax=axes[0],
    showfliers=True,
    linewidth=1.2
)

axes[0].set_title("Vertebrae", fontsize=14)
axes[0].set_xlabel("Vertebral level", fontsize=12)
axes[0].set_ylabel("ASD (mm)", fontsize=12)
axes[0].grid(axis="y", linestyle="--", alpha=0.3)

# -------------------------
# IVDs
# -------------------------

sns.boxplot(
    data=ivd_df,
    x="level",
    y="ASD_mm",
    order=ivd_levels,
    ax=axes[1],
    showfliers=True,
    linewidth=1.2
)

axes[1].set_title("Intervertebral discs", fontsize=14)
axes[1].set_xlabel("Disc level", fontsize=12)
axes[1].set_ylabel("ASD (mm)", fontsize=12)
axes[1].grid(axis="y", linestyle="--", alpha=0.3)

plt.suptitle(
    "Distribution of ASD Across Anatomical Levels",
    fontsize=16,
    y=1.02
)

plt.tight_layout()

plt.savefig(
    "outputs/ASD_distribution_by_level_separate.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()